# 02 - Preprocesamiento

Unimos las fuentes RAW, las limpiamos y construimos el dataset final: **una fila por proveedor y fecha de corte**, con el target ya calculado. Guardamos **un solo archivo**: `data/processed/dataset_proveedor_corte.csv`.

**Target:** `target = 1` si el proveedor recibe una sanción (inhabilitación o multa) en los **12 meses siguientes** a la fecha de corte. Cada predictora usa solo información disponible **hasta** esa fecha.

**Correcciones que dejó el EDA y dónde las resolvemos:**

| Hallazgo del EDA | Qué hacemos | Sección |
|---|---|---|
| Nombres en MAYÚSCULAS y `COL_10` | Encabezados en minúscula; `col_10` pasa a `sector_nivel_gobierno` | 2, 4 |
| Varias filas por contrato | Una fila por contrato-ítem y luego una por contrato | 3, 7 |
| Columnas constantes o sin uso | Leemos solo las columnas necesarias; quitamos las constantes del dataset final | 3, 4, 8 |
| `ruc_contratista` de 6-8 dígitos | Usamos `ruc_destinatario_pago` como identificador | 3 |
| Varias monedas | Homologadas; solo los montos en soles suman monto | 2, 8 |
| Montos muy sesgados | Aplicamos `log1p` | 8 |
| Fechas raras (vigencia hasta 2050, firma antes de la convocatoria) | Pasan a nulo | 7 |
| Fechas y motivos de sanciones | Fechas convertidas; unimos RNP + CONOSCE sin repetir resoluciones | 6 |
| No filtrar información del futuro | Predictoras hasta el corte, target después | 8, 9 |

## 1. Rutas

In [13]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

HORIZONTE_MESES = 12  # el target mira los 12 meses siguientes al corte


def localizar_raiz_repo():
    cwd = Path.cwd().resolve()
    for carpeta in [cwd, *cwd.parents]:
        if (carpeta / "data" / "raw").is_dir():
            return carpeta
    raise FileNotFoundError(
        "No se encontró data/raw. Abre el notebook desde dentro del repositorio descargado. "
        f"Directorio actual: {cwd}"
    )


ROOT = localizar_raiz_repo()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
for carpeta in ["contratos", "convocatorias", "entidades_contratantes", "rnp", "sancionados"]:
    assert (RAW / carpeta).is_dir(), f"Falta la carpeta data/raw/{carpeta}"

print("Raíz del repositorio:", ROOT)
print("Entradas (raw):      ", RAW)
print("Salida (processed):  ", PROCESSED)
print("pandas:", pd.__version__, "| NumPy:", np.__version__)

Raíz del repositorio: D:\CDX\Workspace\G5-IA-APLICADA
Entradas (raw):       D:\CDX\Workspace\G5-IA-APLICADA\data\raw
Salida (processed):   D:\CDX\Workspace\G5-IA-APLICADA\data\processed
pandas: 3.0.6 | NumPy: 2.5.4


## 2. Funciones de limpieza

Las usamos en todas las fuentes: encabezados en minúscula y sin tildes, vacíos a nulo (**no imputamos**), identificadores como texto, fechas y montos a su tipo real, y monedas homologadas.

In [2]:
def sin_tildes(texto):
    s = unicodedata.normalize("NFKD", str(texto))
    return "".join(c for c in s if not unicodedata.combining(c))


def nombre_columna(nombre):
    return re.sub(r"[^a-z0-9]+", "_", sin_tildes(nombre).lower()).strip("_")


def texto_limpio(serie):
    s = serie.astype("string").str.strip()
    return s.mask(s.str.upper().isin(["", "NULL", "NONE", "NAN", "N/A", "<NA>"]))


def mapear_unicos(serie, funcion):
    mapa = {u: funcion(u) for u in serie.dropna().unique()}
    return serie.map(mapa).astype("string")


def limpiar_encabezados(df):
    columnas = [nombre_columna(c) for c in df.columns]
    if len(columnas) != len(set(columnas)):
        raise ValueError("Encabezados duplicados después de normalizar: revisar el archivo de origen.")
    df = df.copy()
    df.columns = columnas
    for c in df.columns:
        df[c] = texto_limpio(df[c])
    return df


def exigir_columnas(df, columnas, archivo):
    faltantes = sorted(set(columnas) - set(df.columns))
    if faltantes:
        raise ValueError(f"{Path(archivo).name}: faltan columnas requeridas {faltantes}.")


def normalizar_id(serie, quitar_ceros=False):
    s = texto_limpio(serie).str.replace(r"\.0+$", "", regex=True)  # quita el .0 que deja Excel
    if quitar_ceros:
        s = s.str.lstrip("0")
        s = s.mask(s.eq("").fillna(False), "0")
    return s


def es_ruc(serie):
    return serie.str.fullmatch(r"\d{11}").fillna(False).astype(bool)


def convertir_fecha(serie, formato=None):
    s = texto_limpio(serie)
    if formato is not None:
        return pd.to_datetime(s, format=formato, errors="coerce").astype("datetime64[ns]")
    iso = s.str.match(r"^\d{4}-\d{2}-\d{2}").fillna(False).astype(bool)
    compacto = s.str.fullmatch(r"\d{8}").fillna(False).astype(bool)
    n = pd.to_numeric(s, errors="coerce")
    serial = (~iso & ~compacto & n.between(20000, 80000)).fillna(False).astype(bool)
    resto = s.notna() & ~iso & ~compacto & ~serial
    salida = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    if iso.any():
        salida.loc[iso] = pd.to_datetime(s[iso], format="ISO8601", errors="coerce").astype("datetime64[ns]")
    if compacto.any():
        salida.loc[compacto] = pd.to_datetime(s[compacto], format="%Y%m%d", errors="coerce").astype("datetime64[ns]")
    if serial.any():
        dias = pd.to_timedelta(n[serial].astype("float64"), unit="D")
        salida.loc[serial] = (pd.Timestamp("1899-12-30") + dias).astype("datetime64[ns]")
    if resto.any():
        salida.loc[resto] = pd.to_datetime(s[resto], format="mixed", dayfirst=True,
                                           errors="coerce").astype("datetime64[ns]")
    return salida


def convertir_monto(serie):
    s = texto_limpio(serie).str.replace(r"^(?:S/\.?|PEN)\s*", "", regex=True, case=False)
    v = pd.to_numeric(s, errors="coerce").astype("float64")
    return v.mask(~np.isfinite(v))


EQUIV_MONEDA = {
    "SOLES": "PEN", "SOL": "PEN", "NUEVOS SOLES": "PEN", "NUEVO SOL": "PEN", "PEN": "PEN",
    "DOLAR NORTEAMERICANO": "USD", "DOLAR AMERICANO": "USD", "DOLARES": "USD", "USD": "USD",
    "EURO": "EUR", "EUR": "EUR", "LIBRA ESTERLINA": "GBP", "DOLAR CANADIENSE": "CAD",
    "FRANCO SUIZO": "CHF", "REAL BRASILENO": "BRL",
}


def normalizar_moneda(serie):
    s = mapear_unicos(texto_limpio(serie), lambda x: sin_tildes(x).upper().strip())
    return mapear_unicos(s, lambda x: EQUIV_MONEDA.get(x, x))


def anio_en_nombre(archivo):
    encontrado = re.search(r"(20\d{2})", Path(archivo).stem)
    return int(encontrado.group(1)) if encontrado else None


def leer_excel_anual(archivo, columnas):
    df = pd.read_excel(archivo, sheet_name=0, dtype="string", engine="openpyxl", keep_default_na=False,
                       usecols=lambda c: nombre_columna(c) in columnas)
    return limpiar_encabezados(df)


def leer_csv_barra(archivo):
    df = pd.read_csv(archivo, sep="|", encoding="cp1252", dtype="string", keep_default_na=False)
    return limpiar_encabezados(df)


def quitar_duplicados(df):
    return df.loc[~df.duplicated(keep="first")].copy()

## 3. Contratos

Apilamos todos los años de `contratos/`. Usamos códigos, montos, moneda, RUC y fechas **de firma** (descartamos adicionales, reducciones y prórrogas porque ocurren después de la firma).

`ruc_contratista` no siempre es un RUC, así que definimos el identificador del proveedor:

| `ruc_contratista` | `tipo_id_proveedor` | `id_proveedor` |
|---|---|---|
| 11 dígitos | `ruc` | el mismo RUC |
| código de 6-8 dígitos | `consorcio_probable` | `ruc_destinatario_pago` |
| `L` + dígitos | `extranjero` | el código `L...` |
| otro | `otro` | el valor original |

In [3]:
COLUMNAS_CONTRATOS = [
    "codigoentidad", "codigoconvocatoria", "n_cod_contrato", "num_item",
    "monto_contratado_total", "monto_contratado_item", "moneda",
    "ruc_contratista", "ruc_destinatario_pago",
    "fecha_publicacion_contrato", "fecha_suscripcion_contrato",
    "fecha_vigencia_inicial", "fecha_vigencia_final",
]


def preprocesar_contratos(archivo):
    df = leer_excel_anual(archivo, set(COLUMNAS_CONTRATOS))
    exigir_columnas(df, COLUMNAS_CONTRATOS, archivo)
    n_entrada = len(df)
    df["codigoentidad_norm"] = normalizar_id(df["codigoentidad"], quitar_ceros=True)
    df["codigoconvocatoria_norm"] = normalizar_id(df["codigoconvocatoria"])
    df["n_cod_contrato_norm"] = normalizar_id(df["n_cod_contrato"])
    df["num_item_norm"] = normalizar_id(df["num_item"])
    for c in ["fecha_publicacion_contrato", "fecha_suscripcion_contrato",
              "fecha_vigencia_inicial", "fecha_vigencia_final"]:
        df[c + "_dt"] = convertir_fecha(df[c])
    for c in ["monto_contratado_total", "monto_contratado_item"]:
        df[c + "_num"] = convertir_monto(df[c])
    df["moneda_norm"] = normalizar_moneda(df["moneda"])
    df["archivo_origen"] = Path(archivo).name
    df["anio_cohorte_archivo"] = anio_en_nombre(archivo)
    df = df.drop(columns=["codigoentidad", "codigoconvocatoria", "n_cod_contrato", "num_item",
                          "monto_contratado_total", "monto_contratado_item", "moneda",
                          "fecha_publicacion_contrato", "fecha_suscripcion_contrato",
                          "fecha_vigencia_inicial", "fecha_vigencia_final"])
    return df, n_entrada


archivos_contratos = sorted((RAW / "contratos").glob("*.xlsx"))
if not archivos_contratos:
    raise FileNotFoundError("No hay archivos .xlsx en data/raw/contratos")

partes, log_contratos = [], []
for archivo in archivos_contratos:
    print("Leyendo:", archivo.name)
    parte, n_entrada = preprocesar_contratos(archivo)
    partes.append(parte)
    log_contratos.append({"archivo": archivo.name, "filas_entrada": n_entrada})
contratos_filas = pd.concat(partes, ignore_index=True)
del partes

ruc_c, ruc_d = contratos_filas["ruc_contratista"], contratos_filas["ruc_destinatario_pago"]
c_ruc = es_ruc(ruc_c)
c_codigo = ruc_c.str.fullmatch(r"\d{6,8}").fillna(False).astype(bool)
c_ext = ruc_c.str.match(r"^L\d+$").fillna(False).astype(bool)
d_ruc = es_ruc(ruc_d)

tipo = np.select([c_ruc, c_codigo & d_ruc, c_codigo, c_ext],
                 ["ruc", "consorcio_probable", "codigo_sin_ruc", "extranjero"], default="otro")
id_prov = np.where(c_ruc, ruc_c.to_numpy(dtype=object),
                   np.where(c_codigo & d_ruc, ruc_d.to_numpy(dtype=object), ruc_c.to_numpy(dtype=object)))
contratos_filas["tipo_id_proveedor"] = pd.Series(tipo, index=contratos_filas.index, dtype="string")
contratos_filas["id_proveedor"] = pd.Series(id_prov, index=contratos_filas.index, dtype="string")

clave = ["n_cod_contrato_norm", "num_item_norm"]
n_filas = len(contratos_filas)
items_contrato = contratos_filas.drop_duplicates(clave, keep="first").reset_index(drop=True)
n_repetidas = n_filas - len(items_contrato)
print(f"Filas de contratos: {n_filas:,} -> contrato-ítem únicos: {len(items_contrato):,}")

resumen_contratos = pd.DataFrame([
    {"concepto": "archivos leídos", "valor": len(archivos_contratos)},
    {"concepto": "filas de entrada (suma de archivos)", "valor": sum(x["filas_entrada"] for x in log_contratos)},
    {"concepto": "filas repetidas (modificaciones del mismo contrato-ítem) descartadas", "valor": n_repetidas},
    {"concepto": "contrato-ítem únicos", "valor": len(items_contrato)},
    {"concepto": "contratos únicos", "valor": items_contrato["n_cod_contrato_norm"].nunique()},
    {"concepto": "fechas de suscripción no interpretables", "valor": int(items_contrato["fecha_suscripcion_contrato_dt"].isna().sum())},
    {"concepto": "fechas de publicación no interpretables", "valor": int(items_contrato["fecha_publicacion_contrato_dt"].isna().sum())},
    {"concepto": "montos totales no interpretables", "valor": int(items_contrato["monto_contratado_total_num"].isna().sum())},
])
display(resumen_contratos)
display(items_contrato["tipo_id_proveedor"].value_counts().rename("filas_contrato_item").to_frame())
display(items_contrato["moneda_norm"].value_counts().rename("filas").to_frame())

Leyendo: CONOSCE_CONTRATOS2023_0.xlsx
Leyendo: CONOSCE_CONTRATOS2024_0.xlsx
Leyendo: CONOSCE_CONTRATOS2025_0.xlsx
Leyendo: CONOSCE_CONTRATOS2026_0.xlsx
Filas de contratos: 31,493 -> contrato-ítem únicos: 22,707


,concepto,valor
0,archivos leídos,4
1,filas de entrada (suma de archivos),31493
2,filas repetidas (modificaciones del mismo cont...,8786
3,contrato-ítem únicos,22707
4,contratos únicos,21342
5,fechas de suscripción no interpretables,0
6,fechas de publicación no interpretables,0
7,montos totales no interpretables,0


,filas_contrato_item
tipo_id_proveedor,
ruc,15174
consorcio_probable,7296
extranjero,232
codigo_sin_ruc,5


,filas
moneda_norm,
PEN,22462
USD,235
EUR,9
GBP,1


## 4. Convocatorias

Apilamos todos los años de `convocatorias/` y dejamos solo las que tienen contrato. Unificamos `COL_10` (2025) como `sector_nivel_gobierno` y descartamos los textos largos.

In [4]:
COLUMNAS_CONVOCATORIAS_OBLIGATORIAS = [
    "codigoentidad", "entidad_ruc", "codigoconvocatoria", "objetocontractual", "tipoprocesoseleccion",
    "tipoentidad", "n_item", "monto_referencial_item", "moneda", "departamento_item", "fecha_convocatoria",
]
COLUMNAS_CONVOCATORIAS = COLUMNAS_CONVOCATORIAS_OBLIGATORIAS + ["sector_nivel_de_gobierno", "col_10"]

codigos_con_contrato = set(items_contrato["codigoconvocatoria_norm"].dropna().unique())


def preprocesar_convocatorias(archivo, codigos):
    df = leer_excel_anual(archivo, set(COLUMNAS_CONVOCATORIAS))
    exigir_columnas(df, COLUMNAS_CONVOCATORIAS_OBLIGATORIAS, archivo)
    n_entrada = len(df)
    df["codigoconvocatoria_norm"] = normalizar_id(df["codigoconvocatoria"])
    df = df.loc[df["codigoconvocatoria_norm"].isin(codigos)].copy()
    sector = df["sector_nivel_de_gobierno"] if "sector_nivel_de_gobierno" in df else pd.Series(pd.NA, index=df.index, dtype="string")
    if "col_10" in df:
        sector = sector.fillna(df["col_10"])
    df["sector_nivel_gobierno"] = mapear_unicos(sector, lambda x: sin_tildes(x).upper())
    df["codigoentidad_conv_norm"] = normalizar_id(df["codigoentidad"], quitar_ceros=True)
    df["entidad_ruc_norm"] = normalizar_id(df["entidad_ruc"])
    df["n_item_norm"] = normalizar_id(df["n_item"])
    df["fecha_convocatoria_dt"] = convertir_fecha(df["fecha_convocatoria"])
    df["monto_referencial_item_num"] = convertir_monto(df["monto_referencial_item"])
    df["moneda_ref_norm"] = normalizar_moneda(df["moneda"])
    for c in ["objetocontractual", "tipoprocesoseleccion", "tipoentidad", "departamento_item"]:
        df[c + "_norm"] = mapear_unicos(df[c], lambda x: sin_tildes(x).upper())
    df = df[["codigoconvocatoria_norm", "n_item_norm", "codigoentidad_conv_norm", "entidad_ruc_norm",
             "objetocontractual_norm", "tipoprocesoseleccion_norm", "tipoentidad_norm",
             "sector_nivel_gobierno", "departamento_item_norm", "fecha_convocatoria_dt",
             "monto_referencial_item_num", "moneda_ref_norm"]]
    return df, n_entrada


archivos_convocatorias = sorted((RAW / "convocatorias").glob("*.xlsx"))
if not archivos_convocatorias:
    raise FileNotFoundError("No hay archivos .xlsx en data/raw/convocatorias")

partes, log_conv = [], []
for archivo in archivos_convocatorias:
    print("Leyendo:", archivo.name, "(puede tardar)")
    parte, n_entrada = preprocesar_convocatorias(archivo, codigos_con_contrato)
    partes.append(parte)
    log_conv.append({"archivo": archivo.name, "filas_entrada": n_entrada, "filas_con_contrato": len(parte)})
convocatorias_items = pd.concat(partes, ignore_index=True)
del partes

convocatorias_items = quitar_duplicados(convocatorias_items)
clave = ["codigoconvocatoria_norm", "n_item_norm"]
repetidas = int(convocatorias_items.duplicated(clave, keep="first").sum())
convocatorias_items = convocatorias_items.drop_duplicates(clave, keep="first").reset_index(drop=True)

display(pd.DataFrame(log_conv))
resumen_convocatorias = pd.DataFrame([
    {"concepto": "convocatoria-ítem conservados (con contrato)", "valor": len(convocatorias_items)},
    {"concepto": "claves (convocatoria, ítem) repetidas descartadas", "valor": repetidas},
    {"concepto": "fechas de convocatoria no interpretables", "valor": int(convocatorias_items["fecha_convocatoria_dt"].isna().sum())},
    {"concepto": "sector/nivel de gobierno vacío", "valor": int(convocatorias_items["sector_nivel_gobierno"].isna().sum())},
])
display(resumen_convocatorias)
display(convocatorias_items["objetocontractual_norm"].value_counts().rename("filas").to_frame())

Leyendo: CONOSCE_CONVOCATORIAS2023_0.xlsx (puede tardar)
Leyendo: CONOSCE_CONVOCATORIAS2024_0.xlsx (puede tardar)
Leyendo: CONOSCE_CONVOCATORIAS2025_0.xlsx (puede tardar)
Leyendo: CONOSCE_CONVOCATORIAS2026_0.xlsx (puede tardar)


,archivo,filas_entrada,filas_con_contrato
0,CONOSCE_CONVOCATORIAS2023_0.xlsx,67459,8582
1,CONOSCE_CONVOCATORIAS2024_0.xlsx,67291,8358
2,CONOSCE_CONVOCATORIAS2025_0.xlsx,63790,5928
3,CONOSCE_CONVOCATORIAS2026_0.xlsx,42217,1151


,concepto,valor
0,convocatoria-ítem conservados (con contrato),24019
1,"claves (convocatoria, ítem) repetidas descartadas",0
2,fechas de convocatoria no interpretables,0
3,sector/nivel de gobierno vacío,0


,filas
objetocontractual_norm,
SERVICIO,8953
BIEN,7741
OBRA,6102
CONSULTORIA DE OBRA,1223


## 5. Entidades contratantes

El código de entidad coincide con el de contratos solo sin ceros iniciales, por eso lo normalizamos así.

In [5]:
archivo_entidades = RAW / "entidades_contratantes" / "entidades_contratantes.csv"
entidades = leer_csv_barra(archivo_entidades)
exigir_columnas(entidades, ["ruc", "codconsucode", "nombre_de_entidad", "departamento", "provincia", "distrito", "estado"], archivo_entidades)
entidades = quitar_duplicados(entidades)
entidades["codigoentidad_norm"] = normalizar_id(entidades["codconsucode"], quitar_ceros=True)
entidades["ruc_entidad_norm"] = normalizar_id(entidades["ruc"])
for c in ["departamento", "provincia", "distrito", "estado"]:
    entidades[c + "_norm"] = mapear_unicos(entidades[c], lambda x: sin_tildes(x).upper())
entidades = entidades[["codigoentidad_norm", "ruc_entidad_norm", "nombre_de_entidad", "departamento_norm",
                       "provincia_norm", "distrito_norm", "estado_norm"]]
repetidos = int(entidades.duplicated("codigoentidad_norm", keep="first").sum())
entidades = entidades.drop_duplicates("codigoentidad_norm", keep="first").reset_index(drop=True)

display(pd.DataFrame([
    {"concepto": "entidades", "valor": len(entidades)},
    {"concepto": "códigos de entidad repetidos descartados", "valor": repetidos},
    {"concepto": "RUC de entidad con formato inválido", "valor": int((~es_ruc(entidades["ruc_entidad_norm"])).sum())},
]))
display(entidades["estado_norm"].value_counts().rename("entidades").to_frame())

,concepto,valor
0,entidades,3366
1,códigos de entidad repetidos descartados,0
2,RUC de entidad con formato inválido,1


,entidades
estado_norm,
ACTIVO,3259
INACTIVO,107


## 6. Sanciones (RNP + CONOSCE)

Unimos inhabilitaciones y multas de ambos orígenes en una tabla de **eventos de sanción**. Clave: `(RUC, resolución, tipo)`; si está en los dos orígenes, dejamos la del RNP. La fecha del evento es el inicio de la inhabilitación o la fecha de resolución de la multa. `fecha_fin` vacía = vigencia indeterminada.

In [6]:
from lxml import html as lxml_html

COLUMNAS_RNP_INH = ["numero", "razon_social", "ruc_proveedor", "resolucion", "periodo",
                    "desde", "hasta", "infraccion", "otra_infraccion", "norma", "estado_actual"]
COLUMNAS_RNP_MULTA = ["numero", "razon_social", "ruc_proveedor", "resolucion", "fecha_resolucion",
                      "monto_multa", "infraccion", "periodo_suspension", "desde", "hasta",
                      "otra_infraccion", "norma", "verificacion_pago", "estado_actual"]


def leer_rnp(archivo):
    documento = lxml_html.fromstring(Path(archivo).read_bytes().decode("cp1252"))
    filas_inh, filas_multa = [], []
    for fila in documento.xpath("//tr"):
        celdas = fila.xpath("./td|./th")
        # 11 celdas = inhabilitaciones, 14 = multas
        if len(celdas) not in (11, 14) or not celdas[0].text_content().strip().isdigit():
            continue
        valores = [" ".join(c.text_content().split()) for c in celdas]
        (filas_inh if len(celdas) == 11 else filas_multa).append(valores)
    if not filas_inh or not filas_multa:
        raise ValueError("No se reconocieron ambas tablas del RNP. Revisar la estructura del .xls.")
    inh = pd.DataFrame(filas_inh, columns=COLUMNAS_RNP_INH)
    multas = pd.DataFrame(filas_multa, columns=COLUMNAS_RNP_MULTA)
    for nombre, tabla in [("inhabilitaciones", inh), ("multas", multas)]:
        numeros = pd.to_numeric(tabla["numero"]).to_numpy(dtype=int)
        if not np.array_equal(np.sort(numeros), np.arange(1, len(tabla) + 1)):
            print(f"Aviso: RNP {nombre}: numeración incompleta o duplicada; revisar la exportación.")
    return inh, multas


archivos_rnp = sorted((RAW / "rnp").glob("*.xls"))
if not archivos_rnp:
    raise FileNotFoundError("No se encontró ningún .xls en data/raw/rnp")
archivo_rnp = archivos_rnp[-1]
print("Exportación RNP usada:", archivo_rnp.name)
rnp_inh, rnp_mul = leer_rnp(archivo_rnp)
rnp_inh, rnp_mul = limpiar_encabezados(rnp_inh), limpiar_encabezados(rnp_mul)

archivo_sanc_inh = RAW / "sancionados" / "sancionados.csv"
archivo_sanc_mul = RAW / "sancionados" / "sancionados_multa.csv"
sanc_inh, sanc_mul = leer_csv_barra(archivo_sanc_inh), leer_csv_barra(archivo_sanc_mul)
exigir_columnas(sanc_inh, ["fecha_corte", "ruc", "fecha_inicio", "fecha_fin", "numero_resolucion"], archivo_sanc_inh)
exigir_columnas(sanc_mul, ["fecha_corte", "ruc", "fecha_inicio", "fecha_fin", "numero_resolucion", "monto"], archivo_sanc_mul)

FMT_RNP, FMT_CONOSCE = "%d/%m/%Y", "%Y%m%d"
mayus = lambda s: mapear_unicos(s, lambda x: x.upper())


def evento(id_proveedor, tipo, resolucion, fecha_evento, inicio, fin, definitiva, monto, motivo, fuente):
    return pd.DataFrame({
        "id_proveedor": normalizar_id(id_proveedor), "tipo": tipo, "resolucion_norm": mayus(texto_limpio(resolucion)),
        "fecha_evento": fecha_evento, "fecha_inicio": inicio, "fecha_fin": fin,
        "es_definitiva": definitiva, "monto_multa": monto, "motivo": motivo, "fuente": fuente,
    })


ini = convertir_fecha(rnp_inh["desde"], FMT_RNP)
e_rnp_inh = evento(rnp_inh["ruc_proveedor"], "inhabilitacion", rnp_inh["resolucion"], ini, ini,
                   convertir_fecha(rnp_inh["hasta"], FMT_RNP),
                   rnp_inh["periodo"].str.upper().eq("DEFINITIVO").fillna(False).astype(bool),
                   np.nan, rnp_inh["infraccion"], "RNP")

e_rnp_mul = evento(rnp_mul["ruc_proveedor"], "multa", rnp_mul["resolucion"],
                   convertir_fecha(rnp_mul["fecha_resolucion"], FMT_RNP),
                   convertir_fecha(rnp_mul["desde"], FMT_RNP), convertir_fecha(rnp_mul["hasta"], FMT_RNP),
                   False, convertir_monto(rnp_mul["monto_multa"]), rnp_mul["infraccion"], "RNP")

ini = convertir_fecha(sanc_inh["fecha_inicio"], FMT_CONOSCE)
e_con_inh = evento(sanc_inh["ruc"], "inhabilitacion", sanc_inh["numero_resolucion"], ini, ini,
                   convertir_fecha(sanc_inh["fecha_fin"], FMT_CONOSCE), False, np.nan,
                   sanc_inh["de_motivo_infraccion"], "CONOSCE")

ini = convertir_fecha(sanc_mul["fecha_inicio"], FMT_CONOSCE)
e_con_mul = evento(sanc_mul["ruc"], "multa", sanc_mul["numero_resolucion"], ini, ini,
                   convertir_fecha(sanc_mul["fecha_fin"], FMT_CONOSCE), False,
                   convertir_monto(sanc_mul["monto"]), sanc_mul["de_motivo_infraccion"], "CONOSCE")

# RNP va antes que CONOSCE: ante una misma resolución se conserva la primera fila
eventos = pd.concat([e_rnp_inh, e_rnp_mul, e_con_inh, e_con_mul], ignore_index=True)
clave_evento = ["id_proveedor", "resolucion_norm", "tipo"]
en_varias_fuentes = eventos.duplicated(clave_evento, keep=False) & ~eventos.duplicated(clave_evento + ["fuente"], keep=False)
n_total_filas = len(eventos)
eventos = eventos.drop_duplicates(clave_evento, keep="first").reset_index(drop=True)
n_sin_fecha = int(eventos["fecha_evento"].isna().sum())
eventos = eventos.loc[eventos["fecha_evento"].notna() & eventos["id_proveedor"].notna()].reset_index(drop=True)

fechas_fuente = [convertir_fecha(sanc_inh["fecha_corte"], FMT_CONOSCE).max(),
                 convertir_fecha(sanc_mul["fecha_corte"], FMT_CONOSCE).max()]
fecha_en_nombre = re.search(r"(\d{4}-\d{2}-\d{2})", archivo_rnp.name)
if fecha_en_nombre:
    fechas_fuente.append(pd.Timestamp(fecha_en_nombre.group(1)))
FECHA_FUENTE_SANCIONES = min(f for f in fechas_fuente if pd.notna(f))
FECHA_INICIO_ETIQUETA_CONFIABLE = eventos["fecha_fin"].min()

print("Fecha de las bases de sanciones (FECHA_FUENTE_SANCIONES):", FECHA_FUENTE_SANCIONES.date())
print("Sanciones anteriores a", FECHA_INICIO_ETIQUETA_CONFIABLE.date(), "están incompletas (FECHA_INICIO_ETIQUETA_CONFIABLE)")

resumen_sanciones = pd.DataFrame([
    {"concepto": "filas de eventos antes de unir fuentes", "valor": n_total_filas},
    {"concepto": "eventos únicos (RUC, resolución, tipo)", "valor": len(eventos) + n_sin_fecha},
    {"concepto": "eventos sin fecha de evento (descartados)", "valor": n_sin_fecha},
    {"concepto": "eventos usados", "valor": len(eventos)},
    {"concepto": "eventos presentes en RNP y CONOSCE", "valor": int(en_varias_fuentes.sum() // 2)},
    {"concepto": "eventos con RUC de 11 dígitos", "valor": int(es_ruc(eventos["id_proveedor"]).sum())},
    {"concepto": "inhabilitaciones con fecha fin vacía (vigencia indeterminada)",
     "valor": int((eventos["tipo"].eq("inhabilitacion") & eventos["fecha_fin"].isna()).sum())},
])
display(resumen_sanciones)
display(eventos.groupby([eventos["fecha_evento"].dt.year.rename("anio_evento"), "tipo"]).size().unstack(fill_value=0).tail(8))

Exportación RNP usada: Reporte_Sancionados_Tribunal_2026-10-11.xls
Fecha de las bases de sanciones (FECHA_FUENTE_SANCIONES): 2026-10-03
Sanciones anteriores a 2021-10-11 están incompletas (FECHA_INICIO_ETIQUETA_CONFIABLE)


,concepto,valor
0,filas de eventos antes de unir fuentes,32019
1,"eventos únicos (RUC, resolución, tipo)",17994
2,eventos sin fecha de evento (descartados),2
3,eventos usados,17992
4,eventos presentes en RNP y CONOSCE,13960
5,eventos con RUC de 11 dígitos,17989
6,inhabilitaciones con fecha fin vacía (vigencia...,1111


tipo,inhabilitacion,multa
anio_evento,,
2019,752,2
2020,606,8
2021,1437,964
2022,1008,1425
2023,1017,1758
2024,1505,837
2025,1919,818
2026,2309,857


## 7. Cruce de fuentes

Cruzamos contrato-ítem con su convocatoria-ítem y con su entidad (`left`, validando que no haya claves repetidas). Luego agregamos a **una fila por contrato**; `fecha_disponible = max(publicación, firma)`.

Pasamos a nulo las fechas imposibles: días entre convocatoria y firma negativos, y duraciones de más de 10 años.

Dejamos solo proveedores con `ruc` o `consorcio_probable`. Los `extranjero` y `codigo_sin_ruc` no pueden aparecer en las listas de sanciones (esas listas usan RUC), así que su target sería 0 por construcción y no por su comportamiento.

In [7]:
items = items_contrato.merge(
    convocatorias_items, how="left",
    left_on=["codigoconvocatoria_norm", "num_item_norm"], right_on=["codigoconvocatoria_norm", "n_item_norm"],
    validate="many_to_one")
items = items.merge(entidades, how="left", on="codigoentidad_norm", validate="many_to_one")

cobertura_cruce = pd.DataFrame([
    {"cruce": "contrato-ítem con convocatoria", "filas": len(items),
     "con_match": int(items["objetocontractual_norm"].notna().sum()),
     "porcentaje": round(items["objetocontractual_norm"].notna().mean() * 100, 2)},
    {"cruce": "contrato-ítem con entidad", "filas": len(items),
     "con_match": int(items["nombre_de_entidad"].notna().sum()),
     "porcentaje": round(items["nombre_de_entidad"].notna().mean() * 100, 2)},
])
display(cobertura_cruce)

ok_ratio = items["monto_contratado_item_num"].notna() & items["monto_referencial_item_num"].gt(0).fillna(False)
items["_monto_item_ok"] = items["monto_contratado_item_num"].where(ok_ratio)
items["_monto_ref_ok"] = items["monto_referencial_item_num"].where(ok_ratio)

contratos = (items.groupby("n_cod_contrato_norm", sort=False).agg(
    id_proveedor=("id_proveedor", "first"),
    tipo_id_proveedor=("tipo_id_proveedor", "first"),
    codigoentidad_norm=("codigoentidad_norm", "first"),
    departamento_entidad=("departamento_norm", "first"),
    objeto=("objetocontractual_norm", "first"),
    tipo_proceso=("tipoprocesoseleccion_norm", "first"),
    sector_nivel_gobierno=("sector_nivel_gobierno", "first"),
    fecha_conv=("fecha_convocatoria_dt", "first"),
    fecha_suscripcion=("fecha_suscripcion_contrato_dt", "first"),
    fecha_publicacion=("fecha_publicacion_contrato_dt", "first"),
    vig_ini=("fecha_vigencia_inicial_dt", "first"),
    vig_fin=("fecha_vigencia_final_dt", "first"),
    monto_total=("monto_contratado_total_num", "first"),
    moneda_norm=("moneda_norm", "first"),
    suma_item=("_monto_item_ok", "sum"),
    suma_ref=("_monto_ref_ok", "sum"),
).reset_index().rename(columns={"n_cod_contrato_norm": "id_contrato"}))

contratos["fecha_disponible"] = contratos[["fecha_suscripcion", "fecha_publicacion"]].max(axis=1)
contratos["monto_pen"] = contratos["monto_total"].where(contratos["moneda_norm"].eq("PEN").fillna(False))
contratos["ratio_contratado_ref"] = (contratos["suma_item"] / contratos["suma_ref"]).where(contratos["suma_ref"] > 0)
contratos["dias_conv_a_firma"] = (contratos["fecha_suscripcion"] - contratos["fecha_conv"]).dt.days.where(lambda s: s >= 0)
contratos["duracion_dias"] = (contratos["vig_fin"] - contratos["vig_ini"]).dt.days.where(lambda s: s.between(0, 3650))
contratos["es_consorcio"] = contratos["tipo_id_proveedor"].eq("consorcio_probable").fillna(False).astype(int)
contratos["es_bien"] = contratos["objeto"].eq("BIEN").fillna(False).astype(int)
contratos["es_servicio"] = contratos["objeto"].eq("SERVICIO").fillna(False).astype(int)
contratos["es_obra"] = contratos["objeto"].eq("OBRA").fillna(False).astype(int)
contratos["es_consultoria_obra"] = contratos["objeto"].eq("CONSULTORIA DE OBRA").fillna(False).astype(int)
contratos["es_contratacion_directa"] = contratos["tipo_proceso"].eq("CONTRATACION DIRECTA").fillna(False).astype(int)
contratos["es_moneda_extranjera"] = (contratos["moneda_norm"].ne("PEN") & contratos["moneda_norm"].notna()).fillna(False).astype(int)
contratos = contratos.drop(columns=["suma_item", "suma_ref"])

poblacion = contratos["tipo_id_proveedor"].isin(["ruc", "consorcio_probable"])
contratos_excluidos = contratos.loc[~poblacion, "tipo_id_proveedor"].value_counts()
contratos_modelo = contratos.loc[poblacion].reset_index(drop=True)

anios_cohorte = items_contrato["anio_cohorte_archivo"].dropna()
INICIO_COBERTURA = pd.Timestamp(int(anios_cohorte.min()), 1, 1)

display(pd.DataFrame([
    {"concepto": "contratos únicos", "valor": len(contratos)},
    {"concepto": "contratos usados (tipos de proveedor permitidos)", "valor": len(contratos_modelo)},
    {"concepto": "proveedores distintos usados", "valor": contratos_modelo["id_proveedor"].nunique()},
    {"concepto": "inicio de cobertura de los datos de contratos", "valor": str(INICIO_COBERTURA.date())},
    {"concepto": "contratos sin fecha de disponibilidad (nunca entran a un corte)", "valor": int(contratos_modelo["fecha_disponible"].isna().sum())},
]))
if len(contratos_excluidos):
    print("Contratos excluidos por tipo de identificador:")
    display(contratos_excluidos.rename("contratos").to_frame())

,cruce,filas,con_match,porcentaje
0,contrato-ítem con convocatoria,22707,22707,100.0
1,contrato-ítem con entidad,22707,22706,100.0


,concepto,valor
0,contratos únicos,21342
1,contratos usados (tipos de proveedor permitidos),21114
2,proveedores distintos usados,12029
3,inicio de cobertura de los datos de contratos,2023-01-01
4,contratos sin fecha de disponibilidad (nunca e...,0


Contratos excluidos por tipo de identificador:


,contratos
tipo_id_proveedor,
extranjero,223
codigo_sin_ruc,5


## 8. Dataset proveedor × fecha de corte y target

- **Cortes:** fin de cada semestre (30-jun y 31-dic), desde que hay datos hasta el último corte cuya ventana de 12 meses ya está completa en las bases de sanciones.
- **Predictoras:** actividad, montos, crecimiento, concentración, diversificación, frecuencia, antigüedad, repetición, composición y antecedentes, siempre hasta el corte.
- **Montos:** solo en soles y en `log1p`.
- Los nulos que quedan (crecimiento, desviación, frecuencia, antecedentes) son esperables y **no se imputan aquí**.

In [8]:
def desplazar(corte, meses):
    return corte - pd.DateOffset(months=meses)


def construir_corte(C, E, corte):
    corte = pd.Timestamp(corte)
    fin_ventana = corte + pd.DateOffset(months=HORIZONTE_MESES)
    # solo contratos disponibles hasta el corte (evita fuga de información)
    H = C.loc[C["fecha_disponible"] <= corte]
    g = H.groupby("id_proveedor")
    F = pd.DataFrame(index=g.size().index)

    F["n_contratos_hist"] = g.size()
    for m in (3, 6, 12, 24):
        gm = H.loc[H["fecha_disponible"] > desplazar(corte, m)].groupby("id_proveedor")
        F[f"n_contratos_{m}m"] = gm.size()
        F[f"monto_pen_{m}m"] = gm["monto_pen"].sum()
    ventanas = [c for c in F.columns if c != "n_contratos_hist"]
    F[ventanas] = F[ventanas].fillna(0)

    est = g["monto_pen"].agg(["sum", "mean", "median", "max", "std"])
    F["monto_pen_hist_total"] = est["sum"]
    F["monto_pen_hist_media"] = est["mean"]
    F["monto_pen_hist_mediana"] = est["median"]
    F["monto_pen_hist_max"] = est["max"]
    F["monto_pen_hist_std"] = est["std"]

    for m in (6, 12):
        ini_prev = desplazar(corte, 2 * m)
        if ini_prev >= INICIO_COBERTURA:
            act = H.loc[H["fecha_disponible"] > desplazar(corte, m)].groupby("id_proveedor")
            prev = H.loc[(H["fecha_disponible"] > ini_prev) & (H["fecha_disponible"] <= desplazar(corte, m))].groupby("id_proveedor")
            n_act = act.size().reindex(F.index, fill_value=0)
            n_prev = prev.size().reindex(F.index, fill_value=0)
            m_act = act["monto_pen"].sum().reindex(F.index, fill_value=0)
            m_prev = prev["monto_pen"].sum().reindex(F.index, fill_value=0)
            F[f"crec_n_{m}m"] = ((n_act - n_prev) / n_prev).where(n_prev > 0)
            F[f"crec_monto_{m}m"] = ((m_act - m_prev) / m_prev).where(m_prev > 0)
        else:
            F[f"crec_n_{m}m"] = np.nan
            F[f"crec_monto_{m}m"] = np.nan

    ge = H.groupby(["id_proveedor", "codigoentidad_norm"]).agg(n=("id_contrato", "size"), monto=("monto_pen", "sum"))
    gp = ge.groupby(level=0)
    F["n_entidades"] = gp.size()
    F["prop_contratos_entidad_principal"] = gp["n"].max() / F["n_contratos_hist"]
    monto_total = gp["monto"].sum()
    F["prop_monto_entidad_principal"] = (gp["monto"].max() / monto_total).where(monto_total > 0)
    F["n_entidades_recurrentes"] = (ge["n"] >= 2).groupby(level=0).sum()
    F["prop_contratos_repetidos_entidad"] = 1 - F["n_entidades"] / F["n_contratos_hist"]
    F["n_departamentos"] = g["departamento_entidad"].nunique()
    F["n_sectores"] = g["sector_nivel_gobierno"].nunique()
    F["n_objetos"] = g["objeto"].nunique()

    fechas = g["fecha_suscripcion"].agg(["min", "max"])
    n = F["n_contratos_hist"]
    F["dias_prom_entre_contratos"] = ((fechas["max"] - fechas["min"]).dt.days / (n - 1)).where(n >= 2)
    F["meses_desde_primer_contrato"] = (corte - fechas["min"]).dt.days / 30.4375
    F["meses_desde_ultimo_contrato"] = (corte - fechas["max"]).dt.days / 30.4375

    for col, nombre in [("es_bien", "prop_bien"), ("es_servicio", "prop_servicio"), ("es_obra", "prop_obra"),
                        ("es_consultoria_obra", "prop_consultoria_obra"),
                        ("es_contratacion_directa", "prop_contratacion_directa"),
                        ("es_moneda_extranjera", "prop_moneda_extranjera")]:
        F[nombre] = g[col].mean()
    F["ratio_contratado_ref_medio"] = g["ratio_contratado_ref"].mean()
    F["dias_conv_a_firma_medio"] = g["dias_conv_a_firma"].mean()
    F["duracion_contrato_dias_medio"] = g["duracion_dias"].mean()

    Ep = E.loc[E["fecha_evento"] <= corte]
    inh_prev = Ep.loc[Ep["tipo"] == "inhabilitacion"]
    mul_prev = Ep.loc[Ep["tipo"] == "multa"]
    F["n_inhab_previas"] = inh_prev.groupby("id_proveedor").size()
    F["n_multas_previas"] = mul_prev.groupby("id_proveedor").size()
    F[["n_inhab_previas", "n_multas_previas"]] = F[["n_inhab_previas", "n_multas_previas"]].fillna(0)
    F["tiene_sancion_previa"] = ((F["n_inhab_previas"] + F["n_multas_previas"]) > 0).astype(int)
    F["dias_desde_ultima_sancion"] = (corte - Ep.groupby("id_proveedor")["fecha_evento"].max()).dt.days
    vigentes = inh_prev.loc[inh_prev["fecha_fin"].isna() | (inh_prev["fecha_fin"] > corte), "id_proveedor"]
    F["inhabilitado_vigente_al_corte"] = F.index.isin(vigentes).astype(int)

    # target: sanciones en (corte, corte + horizonte]
    Ew = E.loc[(E["fecha_evento"] > corte) & (E["fecha_evento"] <= fin_ventana)]
    y_inh = F.index.isin(Ew.loc[Ew["tipo"] == "inhabilitacion", "id_proveedor"])
    y_mul = F.index.isin(Ew.loc[Ew["tipo"] == "multa", "id_proveedor"])
    F[f"y_inhabilitacion_{HORIZONTE_MESES}m"] = y_inh.astype(int)
    F[f"y_multa_{HORIZONTE_MESES}m"] = y_mul.astype(int)
    F["target"] = (y_inh | y_mul).astype(int)

    F["es_consorcio_probable"] = g["es_consorcio"].max().astype(int)
    F["fecha_corte"] = corte

    # montos en escala logarítmica (están muy sesgados)
    for col in [x for x in F.columns if x.startswith("monto_pen_")]:
        F["log_" + col] = np.log1p(F.pop(col).clip(lower=0))
    return F.reset_index()


ultimo_corte = FECHA_FUENTE_SANCIONES - pd.DateOffset(months=HORIZONTE_MESES)
CORTES = [pd.Timestamp(a, m, d)
          for a in range(INICIO_COBERTURA.year, ultimo_corte.year + 1)
          for m, d in [(6, 30), (12, 31)]
          if INICIO_COBERTURA < pd.Timestamp(a, m, d) <= ultimo_corte]
assert min(CORTES) >= FECHA_INICIO_ETIQUETA_CONFIABLE, "Hay cortes con listas de sanciones incompletas"

partes, resumen_cortes = [], []
for corte in CORTES:
    F = construir_corte(contratos_modelo, eventos, corte)
    partes.append(F)
    resumen_cortes.append({
        "fecha_corte": corte.date(), "fin_ventana": (corte + pd.DateOffset(months=HORIZONTE_MESES)).date(),
        "filas": len(F), "positivos": int(F["target"].sum()),
        "prevalencia_%": round(F["target"].mean() * 100, 2),
        "positivos_inhabilitacion": int(F[f"y_inhabilitacion_{HORIZONTE_MESES}m"].sum()),
        "positivos_multa": int(F[f"y_multa_{HORIZONTE_MESES}m"].sum()),
    })

dataset = pd.concat(partes, ignore_index=True)
del partes
resumen_cortes = pd.DataFrame(resumen_cortes)
display(resumen_cortes)

,fecha_corte,fin_ventana,filas,positivos,prevalencia_%,positivos_inhabilitacion,positivos_multa
0,2023-06-30,2024-06-30,1318,33,2.50,20,16
1,2023-12-31,2024-12-31,4144,96,2.32,54,46
2,2024-06-30,2025-06-30,6148,112,1.82,61,53
3,2024-12-31,2025-12-31,8406,208,2.47,124,89
4,2025-06-30,2026-06-30,10033,351,3.50,197,169


### Diccionario de variables

`uso` indica cómo tratar cada columna: **ID**, **Target** o **Predictora**. Verificamos que no falte ni sobre ninguna.

In [9]:
HZ = HORIZONTE_MESES
DICCIONARIO = [
    ("id_proveedor", "Identificación", "RUC del proveedor (o RUC de pago del consorcio probable). Solo para identificar/unir.", "ID"),
    ("fecha_corte", "Identificación", "Fecha de corte: solo se usa información disponible hasta esta fecha.", "ID"),
    ("target", "Target", f"1 si el proveedor tiene una sanción (inhabilitación o multa) con evento en los {HZ} meses posteriores al corte.", "Target"),
    (f"y_inhabilitacion_{HZ}m", "Target", f"1 si recibe una inhabilitación en los {HZ} meses posteriores al corte.", "Target"),
    (f"y_multa_{HZ}m", "Target", f"1 si recibe una multa en los {HZ} meses posteriores al corte.", "Target"),
    ("es_consorcio_probable", "Predictora", "1 si el identificador proviene de un código de consorcio (inferido).", "Predictora"),
    ("n_contratos_hist", "Actividad", "Contratos disponibles hasta el corte (todo el historial observado).", "Predictora"),
]
for m in (3, 6, 12, 24):
    DICCIONARIO += [(f"n_contratos_{m}m", "Actividad", f"Contratos disponibles en los últimos {m} meses antes del corte.", "Predictora"),
                    (f"log_monto_pen_{m}m", "Montos (log1p)", f"Suma de montos contratados en soles en los últimos {m} meses.", "Predictora")]
DICCIONARIO += [
    ("log_monto_pen_hist_total", "Montos (log1p)", "Suma de montos contratados en soles en todo el historial.", "Predictora"),
    ("log_monto_pen_hist_media", "Montos (log1p)", "Monto medio por contrato en soles (historial).", "Predictora"),
    ("log_monto_pen_hist_mediana", "Montos (log1p)", "Monto mediano por contrato en soles (historial).", "Predictora"),
    ("log_monto_pen_hist_max", "Montos (log1p)", "Monto máximo de un contrato en soles (historial).", "Predictora"),
    ("log_monto_pen_hist_std", "Montos (log1p)", "Desviación estándar de los montos por contrato (nulo con un solo contrato).", "Predictora"),
    ("crec_n_6m", "Crecimiento", "Variación relativa del número de contratos: últimos 6 meses vs los 6 anteriores (nulo si el previo es 0 o no hay cobertura).", "Predictora"),
    ("crec_monto_6m", "Crecimiento", "Variación relativa del monto en soles: últimos 6 meses vs los 6 anteriores.", "Predictora"),
    ("crec_n_12m", "Crecimiento", "Variación relativa del número de contratos: últimos 12 meses vs los 12 anteriores.", "Predictora"),
    ("crec_monto_12m", "Crecimiento", "Variación relativa del monto en soles: últimos 12 meses vs los 12 anteriores.", "Predictora"),
    ("n_entidades", "Diversificación", "Entidades distintas con las que contrató.", "Predictora"),
    ("n_departamentos", "Diversificación", "Departamentos distintos de las entidades contratantes.", "Predictora"),
    ("n_sectores", "Diversificación", "Sectores/niveles de gobierno distintos.", "Predictora"),
    ("n_objetos", "Diversificación", "Tipos de objeto distintos (bien, servicio, obra, consultoría de obra).", "Predictora"),
    ("prop_contratos_entidad_principal", "Concentración", "Proporción de contratos con la entidad principal.", "Predictora"),
    ("prop_monto_entidad_principal", "Concentración", "Proporción del monto en soles con la entidad principal.", "Predictora"),
    ("n_entidades_recurrentes", "Repetición", "Entidades con las que firmó 2 o más contratos.", "Predictora"),
    ("prop_contratos_repetidos_entidad", "Repetición", "Proporción de contratos que repiten una entidad ya contratada (1 - entidades/contratos).", "Predictora"),
    ("dias_prom_entre_contratos", "Frecuencia", "Días promedio entre contratos (nulo con un solo contrato).", "Predictora"),
    ("meses_desde_primer_contrato", "Antigüedad", "Meses desde el primer contrato observado hasta el corte (truncado por el inicio de los datos).", "Predictora"),
    ("meses_desde_ultimo_contrato", "Antigüedad", "Meses desde el último contrato hasta el corte.", "Predictora"),
    ("prop_bien", "Composición", "Proporción de contratos de bienes.", "Predictora"),
    ("prop_servicio", "Composición", "Proporción de contratos de servicios.", "Predictora"),
    ("prop_obra", "Composición", "Proporción de contratos de obras.", "Predictora"),
    ("prop_consultoria_obra", "Composición", "Proporción de contratos de consultoría de obra.", "Predictora"),
    ("prop_contratacion_directa", "Composición", "Proporción de contratos por contratación directa.", "Predictora"),
    ("prop_moneda_extranjera", "Composición", "Proporción de contratos en moneda distinta de soles.", "Predictora"),
    ("ratio_contratado_ref_medio", "Proceso", "Monto contratado / monto referencial del ítem, promedio por contrato.", "Predictora"),
    ("dias_conv_a_firma_medio", "Proceso", "Días promedio entre la convocatoria y la firma del contrato.", "Predictora"),
    ("duracion_contrato_dias_medio", "Proceso", "Duración promedio del contrato (vigencia final original − inicial) en días.", "Predictora"),
    ("n_inhab_previas", "Antecedentes", "Inhabilitaciones con evento anterior o igual al corte.", "Predictora"),
    ("n_multas_previas", "Antecedentes", "Multas con evento anterior o igual al corte.", "Predictora"),
    ("tiene_sancion_previa", "Antecedentes", "1 si tiene alguna sanción previa observable.", "Predictora"),
    ("dias_desde_ultima_sancion", "Antecedentes", "Días desde la última sanción previa hasta el corte (nulo si no tiene).", "Predictora"),
    ("inhabilitado_vigente_al_corte", "Antecedentes", "1 si tenía una inhabilitación vigente al corte (fin vacío = vigencia indeterminada).", "Predictora"),
]
diccionario = pd.DataFrame(DICCIONARIO, columns=["columna", "grupo", "descripcion", "uso"])
assert not diccionario["columna"].duplicated().any()
faltan_en_dicc = sorted(set(dataset.columns) - set(diccionario["columna"]))
faltan_en_datos = sorted(set(diccionario["columna"]) - set(dataset.columns))
assert not faltan_en_dicc and not faltan_en_datos, (faltan_en_dicc, faltan_en_datos)
dataset = dataset[diccionario["columna"].tolist()]
display(diccionario.groupby("uso").size().rename("columnas").to_frame())

,columnas
uso,
ID,2
Predictora,44
Target,3


## 9. Validaciones

Si alguna falla, el notebook se detiene: una fila por `(id_proveedor, fecha_corte)`, binarias solo 0/1, ventanas coherentes, ninguna columna constante y recálculo independiente de contratos, antecedentes y target en una muestra (control de fuga de información).

In [10]:
assert not dataset.duplicated(["id_proveedor", "fecha_corte"]).any(), "Hay filas repetidas por proveedor y corte."

binarias = ["target", f"y_inhabilitacion_{HORIZONTE_MESES}m", f"y_multa_{HORIZONTE_MESES}m", "es_consorcio_probable",
            "tiene_sancion_previa", "inhabilitado_vigente_al_corte"]
for col in binarias:
    assert set(dataset[col].unique()) <= {0, 1}, f"{col} no es binaria"
assert (dataset["target"] == (dataset[f"y_inhabilitacion_{HORIZONTE_MESES}m"] | dataset[f"y_multa_{HORIZONTE_MESES}m"])).all()

assert dataset.nunique(dropna=False).gt(1).all(), "Hay columnas constantes"

orden = ["n_contratos_3m", "n_contratos_6m", "n_contratos_12m", "n_contratos_24m", "n_contratos_hist"]
assert (dataset[orden].diff(axis=1).iloc[:, 1:] >= 0).all().all(), "Las ventanas de actividad no son monótonas."

muestra = dataset.sample(min(400, len(dataset)), random_state=42)
for fila in muestra.itertuples(index=False):
    hist = contratos_modelo.loc[(contratos_modelo["id_proveedor"] == fila.id_proveedor)
                                & (contratos_modelo["fecha_disponible"] <= fila.fecha_corte)]
    assert len(hist) == fila.n_contratos_hist, ("Contratos al corte no coinciden", fila.id_proveedor, fila.fecha_corte)
    futuro = eventos.loc[(eventos["id_proveedor"] == fila.id_proveedor)
                         & (eventos["fecha_evento"] > fila.fecha_corte)
                         & (eventos["fecha_evento"] <= fila.fecha_corte + pd.DateOffset(months=HORIZONTE_MESES))]
    assert int(len(futuro) > 0) == fila.target, ("Target no coincide", fila.id_proveedor, fila.fecha_corte)
    previas = eventos.loc[(eventos["id_proveedor"] == fila.id_proveedor) & (eventos["fecha_evento"] <= fila.fecha_corte)]
    assert len(previas) == fila.n_inhab_previas + fila.n_multas_previas, ("Antecedentes no coinciden", fila.id_proveedor)
print(f"Validaciones superadas ({len(muestra)} filas recalculadas de forma independiente).")

nulos = (dataset.isna().mean() * 100).round(1).rename("%_nulos").to_frame()
nulos[nulos["%_nulos"] > 0].sort_values("%_nulos", ascending=False)

Validaciones superadas (400 filas recalculadas de forma independiente).


,%_nulos
dias_desde_ultima_sancion,95.1
crec_monto_12m,82.5
crec_n_12m,82.4
log_monto_pen_hist_std,76.9
dias_prom_entre_contratos,76.7
crec_monto_6m,70.3
crec_n_6m,70.2
log_monto_pen_hist_media,0.4
log_monto_pen_hist_max,0.4
log_monto_pen_hist_mediana,0.4


## 10. Guardar el archivo

Escribimos **un solo archivo** (la carpeta se crea si no existe).

Imputación, outliers, escalado y SMOTE **no se hacen aquí**: van en el notebook de modelado, después de dividir train/test por tiempo y ajustados solo con train, para no filtrar información del futuro.

In [14]:
PROCESSED.mkdir(parents=True, exist_ok=True)
ruta_salida = PROCESSED / "dataset_proveedor_corte.csv"
dataset.to_csv(ruta_salida, index=False)

print("Archivo guardado:", ruta_salida)
print(f"Filas: {len(dataset):,} | Columnas: {len(dataset.columns)} | Tamaño: {ruta_salida.stat().st_size / 1e6:.1f} MB")

Archivo guardado: D:\CDX\Workspace\G5-IA-APLICADA\data\processed\dataset_proveedor_corte.csv
Filas: 30,049 | Columnas: 49 | Tamaño: 9.6 MB
